# Potential Flow

AE2130-I, Delft University of Technology.

In [16]:
# page:hide
%pip install -q plotly ipywidgets

import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [17]:
# page:hide
from scipy.interpolate import RectBivariateSpline
from scipy.interpolate import RegularGridInterpolator

def rk4_step(f, xn, h):
    """One classical RK4 step for dx/ds = f(x)."""
    k1 = f(xn)
    k2 = f(xn + h/2*k1)
    k3 = f(xn + h/2*k2)
    k4 = f(xn + h*k3)
    return xn + h/6*(k1 + 2*k2 + 2*k3 + k4)

def _trace(f, x0, ds, n_max, inside):
    """March from x0 with RK4 steps of size ds (negative = backwards)."""
    pts = [np.asarray(x0, float)]
    for _ in range(n_max):
        xn  = pts[-1]
        xn1 = rk4_step(f, xn, ds)
        if not inside(xn1) or np.allclose(xn1, xn):
            break
        pts.append(xn1)
    return np.array(pts)

def streamline(x0, uv, xy, ds=0.01, n_fwd=300, n_back=300):
    (u, v), (x, y) = uv, xy
    U = RectBivariateSpline(x[::2], y[::2], u[::2, ::2], kx=3, ky=3)   # cubic interpolation
    V = RectBivariateSpline(x[::2], y[::2], v[::2, ::2], kx=3, ky=3)

    def f(xn):                                     # unit velocity direction
        vel = np.array([U(*xn)[0, 0], V(*xn)[0, 0]])
        sp = np.linalg.norm(vel)
        return vel / sp if sp > 1e-12 else np.zeros(2)

    def inside(xn):
        return x[0] <= xn[0] <= x[-1] and y[0] <= xn[1] <= y[-1]

    fwd  = _trace(f, x0,  ds, n_fwd,  inside)
    back = _trace(f, x0, -ds, n_back, inside)
    return np.vstack([back[::-1], fwd[1:]]).T     # back … x0 … forward


This notebook serves to showcase the potential flow module taught in AE2130-I, which follows Anderson's *Fundamentals of Aerodynamics* [@anderson2017]. It is an interactive notebook, so you can modify the code yourself!

**A note on notation.** Throughout this notebook we work in 2D. The divergence is written with a vector operator, $\vec{\nabla}\cdot\vec{u}$. The rotation, however, has only one component in 2D, so we treat it as a scalar and drop the arrow:

$$
\begin{equation}
    \nabla\times\vec{u} = \frac{\partial v}{\partial x} - \frac{\partial u}{\partial y} \qquad\text{scalar rotation in 2D}
\end{equation}
$$

## Elementary Flows

### Uniform Flow
<!-- page:closed -->

Let's start with the simplest elementary flow. Consider a flow at an angle $\alpha$ with components

$$
\vec{u} = \begin{pmatrix} u \\ v \end{pmatrix}
        = \begin{pmatrix} V_{\infty}\cos\alpha \\ V_{\infty}\sin\alpha \end{pmatrix}.
$$

This is a **uniform flow**: the velocity is the same everywhere in the domain. It is both divergence free, $\vec{\nabla}\cdot\vec{u} = 0$, and irrotational, $\nabla\times\vec{u} = 0$. The velocity potential $\phi$ must satisfy

$$
\frac{\partial \phi}{\partial x} = u = V_{\infty}\cos\alpha,
\qquad
\frac{\partial \phi}{\partial y} = v = V_{\infty}\sin\alpha.
$$

Integrating each equation gives

$$
\begin{array}{r@{\;}c@{\;}c@{\;}c@{\;}l}
\phi(x,y) = & V_{\infty}\, x\cos\alpha & + & C(y)         & +\; C, \\[6pt]
            & \Big\downarrow           &   & \Big\uparrow & \\[6pt]
\phi(x,y) = & C(x)                     & + & V_{\infty}\, y\sin\alpha & +\; C.
\end{array}
$$

The integration constant of the first line can still depend on $y$, and that of the second on $x$. Both integrals describe the same $\phi$, so they must match term by term: $C(x) = V_{\infty}\, x\cos\alpha$ and $C(y) = V_{\infty}\, y\sin\alpha$. Hence

$$
\phi(x,y) = V_{\infty}\left(x\cos\alpha + y\sin\alpha\right) + C.
$$

We can do the same for the stream function, using $u = \partial\psi/\partial y$ and $v = -\partial\psi/\partial x$. Dropping the constants, the uniform flow is then described by

$$
\boxed{
\begin{aligned}
\phi(x,y) &= V_{\infty} \Big( x\cos\alpha + y\sin\alpha \Big) \\[6pt]
\psi(x,y) &= V_{\infty} \Big( y\cos\alpha - x\sin\alpha \Big)
\end{aligned}
}
$$

Drawing iso-contours of the two equations would show:

```tikz
\usetikzlibrary{arrows.meta}
\begin{tikzpicture}[>={Stealth[length=2.2mm]}, line width=0.6pt]
  % streamlines (psi = const), each with an arrow mid-way and at the end
  \foreach \y in {0,1.6,3.2,4.8,6.4} {
    \draw[->] (0,\y) -- (3.4,\y);
    \draw[->] (3.4,\y) -- (7,\y);
  }
  % equipotential line (phi = const); the white box hides the streamline behind its label
  \draw[dashed] (5.6,-0.6) -- (5.6,7);
  \node[rotate=90, fill=white, inner sep=2pt] at (5.25,3.1) {$\phi = \mathrm{const}$};
  % labels
  \node[above] at (3.3,1.6) {$\phantom{\mathrm{const}}\psi = \mathrm{const}\phantom{\psi}$}; % phantoms centre the '=' over the arrow
  \node[above] at (3.4,3.2) {$V_\infty$};
\end{tikzpicture}
```

### Source Flow
<!-- page:closed -->

We now consider our next elementary flow component. Suppose we have an origin point, and from this origin point, flow moves out in a radial fashion at a rate $u_r$ without any rotational motion $u_\theta$, like the following image:

```tikz
\usetikzlibrary{arrows.meta}
\begin{tikzpicture}[>={Stealth[length=2.2mm]}, line width=0.6pt]
  % streamlines: straight rays out of the origin, each with an arrow mid-way and at the end
  \foreach \a in {0,30,...,330} {
    \draw[->] (0,0) -- (\a:1.6);
    \draw[->] (\a:1.6) -- (\a:3);
  }
  % r and theta, as in the source/sink figure
  \draw[->, line width=0.4pt] (0.7,0) arc (0:30:0.7);
  \node at (15:0.95) {$\theta$};
  \node[above left] at (30:2.15) {$r$};
  % the streamlines themselves show u_r: label one at its tip
  \node[right] at (30:3) {$u_r$};
  \draw[fill=white, line width=1pt] (0,0) circle (3pt);
\end{tikzpicture}
```

This is as if you are generating a flow from a source, hence we call this a **source flow**. From a practical perspective, this is as if a nozzle is injecting fluid into your domain. If you switch the flow direction, we call it a **sink**, and that can be viewed intuitively like your kitchen drain. Lets start by analysing the mass flux $\dot{m}$ through the circle at a fixed $r$:

```tikz
\usetikzlibrary{arrows.meta}
\begin{tikzpicture}[>={Stealth[length=2.2mm]}, line width=0.6pt]
  % control volume: the disc Omega of radius r, bounded by the dashed circle dOmega
  \draw[dashed] (0,0) circle (2);
  % radius r
  \draw[line width=0.4pt] (0,0) -- (45:2);
  \node[above left=-1pt] at (45:1) {$r$};
  % outward normal n and tangent t at a point on the boundary
  \draw[->, line width=1pt] (45:2) -- ++(45:1.1) node[above right=-2pt] {$\vec{n}$};
  \draw[->, line width=1pt] (45:2) -- ++(135:1.1) node[above left=-2pt] {$\vec{t}$};
  % labels
  % opposite r, n and t: Omega inside the circle, dOmega just outside it
  \node at (225:1) {$\Omega$};
  \node at (225:2.45) {$\partial\Omega$};
  \draw[fill=white, line width=1pt] (0,0) circle (3pt);
\end{tikzpicture}
```
THIS IS FIGURE XX

$$
\begin{equation}
    \dot{m} = \oint_{\partial\Omega} (\rho\vec{u})\cdot\vec{n}\,\mathrm{d}l = \int\limits_0^{2\pi} \rho u_r r\,\mathrm{d}\theta = \rho r u_r \int\limits_0^{2\pi} \mathrm{d}\theta = 2\pi \rho r u_r
\end{equation}
$$

rearranging gives us:

$$
\begin{equation}
    u_r = \frac{\dot{m}/\rho}{2\pi r}
\end{equation}
$$

and setting $\Lambda = \dot{m}/\rho$, we have sufficient information to describe the velocity field associated with a source flow:

$$
\begin{equation}
    u_r = \frac{\Lambda}{2\pi r}
\end{equation}
$$
$$
\begin{equation}
    u_\theta = 0
\end{equation}
$$

Of particular note is that the $1/r$ decay leads to an incompressible flow:

$$
\begin{equation}
    \vec{\nabla}\cdot\vec{u} = \frac{1}{r}\frac{\partial(r u_r)}{\partial r} + \frac{1}{r} \frac{\partial u_\theta}{\partial \theta} = 0\, \qquad r > 0
\end{equation}
$$

and it is by construction, irrotational:

$$
\begin{equation}
    \nabla\times\vec{u} = \frac{1}{r}\frac{\partial(r u_\theta)}{\partial r} - \frac{1}{r} \frac{\partial u_r}{\partial \theta} = 0\, \qquad r > 0
\end{equation}
$$

We want to now determine the potential, thus:

$$
\begin{alignat}{3}
    u_r      &\;\;=\;\;& \frac{\partial\phi}{\partial r}                &\;\;=\;\;&& \frac{\Lambda}{2\pi r} \\[6pt]
    u_\theta &\;\;=\;\;& \frac{1}{r}\frac{\partial\phi}{\partial\theta} &\;\;=\;\;&& 0
\end{alignat}
$$

integrating both yields:

$$
\begin{array}{r@{\;}c@{\;}c@{\;}c@{\;}l}
\phi(r,\theta) = & \dfrac{\Lambda}{2\pi}\ln{r} & + & C(\theta)    & +\; C, \\[6pt]
                 & \Big\downarrow              &   & \Big\uparrow & \\[6pt]
\phi(r,\theta) = & C(r)                        & + & 0            & +\; C.
\end{array}
$$

In similar fashion to how we did the uniform flow:

$$
\begin{equation}
    \phi(r,\theta) = \frac{\Lambda}{2\pi} \ln{r}
\end{equation}
$$


We can do the same for the stream function, using $u_r = 1/r\; \partial\psi/\partial\theta$ and $u_\theta = -\partial\psi/\partial r$. Dropping the constants, the source flow is then described by

$$
\boxed{
\begin{aligned}
\phi(r,\theta) &= \frac{\Lambda}{2\pi} \ln{r} \\[6pt]
\psi(r,\theta) &= \frac{\Lambda}{2\pi} \theta
\end{aligned}
}
$$

Drawing iso-contours of the two equations would show:

```tikz
\usetikzlibrary{arrows.meta}
\begin{tikzpicture}[>={Stealth[length=2.2mm]}, line width=0.6pt]
  % common parts of both drawings: dashed equipotential circle (phi = const), r and theta
  \foreach \cx in {0,7.5} {
    \begin{scope}[shift={(\cx,0)}]
      \draw[dashed] (0,0) circle (1.2);
      \draw[->, line width=0.4pt] (0.7,0) arc (0:30:0.7);
      \node at (15:0.95) {$\theta$};
      \node[above left] at (30:2.15) {$r$};
    \end{scope}
  }
  % source: radial streamlines (psi = const) every 30 degrees, flowing outward
  \foreach \a in {0,30,...,330} {
    \draw[->] (0,0) -- (\a:1.6);
    \draw[->] (\a:1.6) -- (\a:3);
  }
  \node[right] at (30:3) {$u_r$};
  \draw[fill=white, line width=1pt] (0,0) circle (3pt);
  % sink: the same streamlines, flowing inward
  \begin{scope}[shift={(7.5,0)}]
    \foreach \a in {0,30,...,330} {
      \draw[->] (\a:3) -- (\a:1.6);
      \draw[->] (\a:1.6) -- (\a:0.75);
      \draw (\a:0.75) -- (0,0);
    }
    \node[right] at (30:3) {$-u_r$};
    \draw[fill=white, line width=1pt] (0,0) circle (3pt);
  \end{scope}
  % labels
  \node (phi) at (-2.4,0.5) {$\phi = \mathrm{const}$};
  \draw[line width=0.4pt] (phi.east) -- (158:1.2);
  \node[above, rotate=60] at (60:2.3) {$\psi = \mathrm{const}$};
  \node at (0,-3.6) {Source flow};
  \node at (7.5,-3.6) {Sink flow};
\end{tikzpicture}
```

A good question to ask is if a source flow is truly incompressible and irrotational everywhere. After all, the source point itself is a singularity: $u_r \rightarrow \infty$, by definition. Instead of looking at incompressibility and irrotationality _pointwise_, we can look at it from _an integral perspective_. We begin by creating a control volume that encompasses our singularity, which we conveniently done already in FIGURE XX
Consider the volume flux through $\partial\Omega$:

$$
\begin{equation}
    \oint_{\partial\Omega} \vec{u}\cdot\vec{n} \,\mathrm{d}l = \int\limits_0^{2\pi} u_r r \,\mathrm{d}\theta = \int\limits_0^{2\pi} \frac{\Lambda}{2\pi r}r \,\mathrm{d}\theta = \Lambda
\end{equation}
$$
which is independent of $r$, so it holds true however small the control volume is. Now recall the divergence theorem (for 2D):
$$
\begin{equation}
    \oint_{\partial\Omega} \vec{u}\cdot\vec{n} \,\mathrm{d}l = \int_\Omega \vec{\nabla}\cdot\vec{u}\,\mathrm{d}A
\end{equation}
$$
thus:
$$
\begin{equation}
    \int_\Omega \vec{\nabla}\cdot\vec{u}\,\mathrm{d}A = \Lambda
\end{equation}
$$

**This means that our problem is divergence free everywhere except at the source point.** Now consider the circulation around the exact same boundary:

$$
\begin{equation}
    \oint_{\partial\Omega} \vec{u}\cdot\vec{t}\,\mathrm{d}l = \int\limits_0^{2\pi} u_\theta r\,\mathrm{d}\theta = 0
\end{equation}
$$
as $u_\theta = 0$. This holds true regardless of $r$. Recall Stokes' theorem (for 2D):

$$
\begin{equation}
    \oint_{\partial\Omega} \vec{u}\cdot\vec{t}\,\mathrm{d}l = \int_\Omega \nabla\times\vec{u} \,\mathrm{d}A
\end{equation}
$$
thus:
$$
\begin{equation}
    \int_\Omega \nabla\times\vec{u} \,\mathrm{d}A = 0
\end{equation}
$$

**This entails that the flow is irrotational everywhere, including the singularity, in an integral sense.**


### Doublet Flow
<!-- page:closed -->

```tikz
\usetikzlibrary{arrows.meta}
\begin{tikzpicture}[>={Stealth[length=2.2mm]}, line width=0.6pt]
  % axes
  \draw (-3.5,0) -- (3.5,0);
  \draw (0,-3.4) -- (0,3.4);
  % streamlines: circles of diameter kappa/(2 pi c), tangent to the x axis at the origin.
  % The flow leaves the origin towards -x and comes back in from +x.
  \foreach \R in {0.8,1.3,1.8} {
    \foreach \s in {1,-1} {
      \draw (0,\s*\R) circle (\R);
      \draw[->] (-\R,\s*\R+0.01*\s) -- (-\R,\s*\R+0.02*\s);  % left side: moving away from the x axis
      \draw[->] (\R,\s*\R-0.01*\s) -- (\R,\s*\R-0.02*\s);    % right side: moving back towards it
    }
  }
  % equipotential lines (phi = const): circles tangent to the y axis at the origin
  \foreach \R in {0.5,0.85,1.2} {
    \draw[dashed] (\R,0) circle (\R);
    \draw[dashed] (-\R,0) circle (\R);
  }
  \node (phi) at (3.3,-2.4) {$\phi = \mathrm{const}$};
  \draw[line width=0.4pt] (phi.north west) -- (1.97,-0.92);
  % doublet strength: arrow along -x
  \draw[->, line width=1pt] (0,0) -- (-1.1,0);
  \node[left, fill=white, inner sep=1pt] at (-1.15,0) {$\kappa$};
  \draw[fill=white, line width=1pt] (0,0) circle (3pt);
  % diameter of the largest circle
  \draw[<->, line width=0.4pt] (-2.6,0) -- (-2.6,3.6);
  \draw[line width=0.4pt] (-2.75,3.6) -- (0,3.6);
  \node[left] at (-2.6,1.8) {$\displaystyle\frac{\kappa}{2\pi c}$};
\end{tikzpicture}
```

### Vortex Flow
<!-- page:closed -->

```tikz
\usetikzlibrary{arrows.meta}
\begin{tikzpicture}[>={Stealth[length=2.2mm]}, line width=0.6pt]
  % streamlines: concentric circles, flow counter-clockwise
  \foreach \R in {1,1.9,2.8} {
    \draw (0,0) circle (\R);
    \foreach \a in {0,90,180,270} {
      \draw[->] (\a+59:\R) arc (\a+59:\a+61:\R);
    }
  }
  % equipotential lines (phi = const): rays out of the origin.
  % The 45 degree one stops inside the outer circle so it doesn't cross the u_theta arrow.
  \foreach \a in {0,90,135,...,315} {
    \draw[dashed] (\a:0.3) -- (\a:3.3);
  }
  \draw[dashed] (45:0.3) -- (45:2.5);
  \node[above, rotate=-45] at (135:3.75) {$\phi = \mathrm{const}$};
  % radius to the outer circle and the tangential velocity there
  \draw[->] (0,0) -- (25:2.8);
  \node[below right] at (25:1.5) {$r$};
  \draw[->, line width=1pt] (25:2.8) -- ++(115:1.3) node[above] {$u_\theta$};
  \draw[fill=white, line width=1pt] (0,0) circle (3pt);
\end{tikzpicture}
```

## Superimposed Flows

We now make an effort to displaying a series of combinations of potential flows

In [18]:
# page:fold
x = np.linspace(-1,1,200)
y = np.linspace(-1,1,200)

X, Y     = np.meshgrid(x,y, indexing='ij')
R, THETA = np.hypot(X,Y), np.arctan2(Y,X)

In [19]:
# page:fold
def uniform(Vinf,angle, x=X,y=Y):
    Vx, Vy = Vinf*np.cos(angle), Vinf*np.sin(angle)

    phi = Vx*x + Vy*y
    psi = Vx*y - Vy*x

    return phi, psi

def source(Lambda, r=R,theta=THETA):
    strength = Lambda/(2*np.pi)

    phi = strength*np.log(r)
    psi = strength*theta

    return phi, psi

def doublet(kappa, r=R,theta=THETA):
    strength = kappa/(2*np.pi)

    phi =  strength*np.cos(theta)/r
    psi = -strength*np.sin(theta)/r

    return phi, psi

def vortex(Gamma, r=R,theta=THETA):
    strength = Gamma/(2*np.pi)

    phi = -strength*theta
    psi =  strength*np.log(r)

    return phi, psi

In [20]:
# page:fold
def ddx(f,x=x): return np.gradient(f, x, axis=0)
def ddy(f,x=y): return np.gradient(f, y, axis=1)

### Semi-infinite Body
<!-- page:closed -->

_To be written._

In [21]:
# page:hide
# page:rerun=1

# page:slider (0.5, 5.0, 0.1)
Vinf = 1

# page:slider (-30, 30, 1)
angle = 0

# page:slider (1, 5, 0.1)
Lambda = 1

In [22]:
phi_1, psi_1 = uniform(Vinf,np.deg2rad(angle))
phi_2, psi_2 = source(Lambda)

phi_t = phi_1 + phi_2
psi_t = psi_1 + psi_2

u = ddx(phi_t)
v = ddy(phi_t)

V  = np.hypot(u, v)
Cp = 1 - (V/Vinf)**2

# page:hide-from
fig = go.Figure()
def draw_streamline(x0, width=1, **opts):
    xs, ys = streamline(x0, (u, v), (x, y), **opts)
    fig.add_scatter(x=xs, y=ys, mode="lines", line=dict(color="white", width=width), hoverinfo="skip")

## -------------- ##
## Pressure Field ##
## -------------- ##
dark_rdbu_r = [[0.0, "#4a86c5"],    # light blue  (Cp = -1, suction)
               [0.5, "#1a1a1a"],   # white       (Cp = 0)
               [1.0, "#d9605f"]]   # light red   (Cp = 1, stagnation)
fig.add_contour(x=x, y=y, z=Cp.T,
                contours=dict(start=-1, end=1, size=0.125, coloring="fill"),
                line_width=0, colorscale=dark_rdbu_r, zmin=-1, zmax=1, zmid=0,
                colorbar=dict(title="Cp"), hoverinfo="skip")

## ----------------------- ##
## free-stream streamlines ##
## ----------------------- ##
a  = np.deg2rad(angle)
c  = -0.9*np.array([np.cos(a), np.sin(a)])       # centre of the line, upstream
e  =  0.9*np.array([-np.sin(a), np.cos(a)])      # half-length along the perpendicular
seeds = np.linspace(c - e, c + e, 20)            # (21, 2) points from one end to the other

for x0 in seeds:
    draw_streamline(x0, width=1)

## ------------------ ##
## source streamlines ##
## ------------------ ##
R, n = 0.05, 8                                   # circle radius, number of seeds
ang = np.linspace(0, 2*np.pi, n, endpoint=False)
ring = np.array([0.0, 0.0]) + R*np.column_stack([np.cos(ang), np.sin(ang)])

for x0 in ring:
    draw_streamline(x0, width=1)
fig.add_scatter(x=[0], y=[0], mode="markers", marker=dict(size=10, color="red"), name="source")

## ---------------- ##
## stagnation point ##
## ---------------- ##
a = np.deg2rad(angle)
d = Lambda / (2*np.pi*Vinf)
t      = np.array([np.cos(a), np.sin(a)])     # freestream direction
nrm    = np.array([-np.sin(a), np.cos(a)])    # perpendicular to it
x_stag = -d * t                               # stagnation point
eps = 0.01

seeds = [(x_stag - eps*t,   dict(n_fwd=0)),   # incoming arm: trace backwards upstream
         (x_stag + eps*nrm, dict(n_back=0)),  # upper branch: trace forwards
         (x_stag - eps*nrm, dict(n_back=0))]  # lower branch: trace forwards

for x0, opts in seeds:
    draw_streamline(x0, width=3, **opts)

fig.add_scatter(x=[x_stag[0]], y=[x_stag[1]], mode="markers",
                marker=dict(symbol="star", size=16, color="black",
                            line=dict(color="white", width=1.5)),
                showlegend=False)

fig.update_layout(title="Uniform flow + source", width=700, height=600, showlegend=False,
                  xaxis=dict(range=[-1, 1], constrain="domain"), yaxis=dict(range=[-1, 1], scaleanchor="x"))
fig.update_layout(template="plotly_dark",
                  paper_bgcolor="rgba(0,0,0,0)",   # transparent: shows the page's black
                  plot_bgcolor="rgba(0,0,0,0)",
                  font_color="#e0e0e0")
fig.update_xaxes(fixedrange=True)          # no zooming or panning on any axis
fig.update_yaxes(fixedrange=True)
fig.update_layout(dragmode=False)          # dragging doesn't box-select either
fig.show(config={"displayModeBar": False, "scrollZoom": False})
# page:hide-to

### Rankine Oval
<!-- page:closed -->

_To be written._

In [23]:
# page:hide
# page:rerun=1

# page:slider (0.5, 5.0, 0.1)
Vinf = 1

# page:slider (0.05, 1.0, 0.05)
sep  = 1.0

# page:slider (1, 5, 0.1)
Lambda = 1

In [24]:
R2, THETA2 = np.hypot(X+sep/2,Y), np.arctan2(Y,X+sep/2)
R3, THETA3 = np.hypot(X-sep/2,Y), np.arctan2(Y,X-sep/2)

phi_1, psi_1 = uniform(Vinf,0)
phi_2, psi_2 = source( Lambda, r=R2, theta=THETA2)
phi_3, psi_3 = source(-Lambda, r=R3, theta=THETA3)

phi_t = phi_1 + phi_2 + phi_3
psi_t = psi_1 + psi_2 + psi_3

u = ddx(phi_t)
v = ddy(phi_t)

V  = np.hypot(u, v)
Cp = 1 - (V/Vinf)**2

# page:hide-from
fig = go.Figure()
def draw_streamline(x0, width=1, **opts):
    xs, ys = streamline(x0, (u, v), (x, y), **opts)
    fig.add_scatter(x=xs, y=ys, mode="lines", line=dict(color="white", width=width), hoverinfo="skip")

## -------------- ##
## Pressure Field ##
## -------------- ##
dark_rdbu_r = [[0.0, "#4a86c5"],    # blue (Cp = -1, suction)
               [0.5, "#1a1a1a"],    # dark (Cp = 0)
               [1.0, "#d9605f"]]    # red  (Cp = 1, stagnation)
fig.add_contour(x=x, y=y, z=Cp.T,
                contours=dict(start=-1, end=1, size=0.125, coloring="fill"),
                line_width=0, colorscale=dark_rdbu_r, zmin=-1, zmax=1,
                colorbar=dict(title="Cp"), hoverinfo="skip")

## ---------------------------------- ##
## free-stream + internal streamlines ##
## ---------------------------------- ##
for y0 in np.linspace(-1, 1, 21)[1:-1]:    # seeds on the line x = 0, traced both ways
    draw_streamline(np.array([0.0, y0]))

## ----------------- ##
## stagnation points ##
## ----------------- ##
d = np.sqrt((sep/2)**2 + Lambda*(sep/2)/(np.pi*Vinf))   # distance from the centre
front, rear = np.array([-d, 0.0]), np.array([d, 0.0])
eps = 0.01

seeds = [(front + np.array([-eps, 0]), dict(n_fwd=0)),   # incoming arm: trace backwards upstream
         (front + np.array([0,  eps]), dict(n_back=0)),  # upper half of the oval
         (front + np.array([0, -eps]), dict(n_back=0)),  # lower half of the oval
         (rear  + np.array([ eps, 0]), dict(n_back=0))]  # outgoing arm: trace forwards downstream

for x0, opts in seeds:
    draw_streamline(x0, width=3, **opts)

fig.add_scatter(x=[front[0], rear[0]], y=[0, 0], mode="markers",
                marker=dict(symbol="star", size=16, color="black",
                            line=dict(color="white", width=1.5)))
fig.add_scatter(x=[-sep/2, sep/2], y=[0, 0], mode="markers",
                marker=dict(size=10, color=["red", "blue"]))   # source red, sink blue

fig.update_layout(title="Rankine oval: uniform flow + source + sink", width=700, height=600,
                  showlegend=False,
                  xaxis=dict(range=[-1, 1], constrain="domain"),
                  yaxis=dict(range=[-1, 1], scaleanchor="x"))
fig.update_layout(template="plotly_dark",
                  paper_bgcolor="rgba(0,0,0,0)", plot_bgcolor="rgba(0,0,0,0)",
                  font_color="#e0e0e0")
fig.update_xaxes(fixedrange=True)          # no zooming or panning on any axis
fig.update_yaxes(fixedrange=True)
fig.update_layout(dragmode=False)          # dragging doesn't box-select either
fig.show(config={"displayModeBar": False, "scrollZoom": False})
# page:hide-to

### Non-lifting flow over a cylinder
<!-- page:closed -->

You can see as you reduce the separation $s$ while increasing $\Lambda$ such that $s\Lambda = \mathrm{const}$, that the Rankine oval becomes more-and-more like a circle!

In [25]:
# page:hide
# page:rerun=1

# page:slider (0.5, 5.0, 0.1)
Vinf = 1

# page:slider (0.1, 0.6, 0.05)
R = 0.4

In [26]:
kappa = 2*np.pi*Vinf*R**2                            # doublet strength that gives a cylinder of radius R

phi_1, psi_1 = uniform(Vinf, 0)
phi_2, psi_2 = doublet(kappa)

phi_t = phi_1 + phi_2
psi_t = psi_1 + psi_2

u = ddx(phi_t)
v = ddy(phi_t)

V  = np.hypot(u, v)
Cp = 1 - (V/Vinf)**2

# page:hide-from
Cp_plot = np.where(np.hypot(X, Y) < R, np.nan, Cp)   # blank out the inside of the cylinder

fig = make_subplots(rows=1, cols=2, column_widths=[0.5, 0.5], horizontal_spacing=0.22,
                    subplot_titles=("Flow field", "Surface pressure"))

def draw_streamline(x0, width=1, **opts):
    xs, ys = streamline(x0, (u, v), (x, y), **opts)
    fig.add_scatter(x=xs, y=ys, mode="lines", line=dict(color="white", width=width),
                    hoverinfo="skip", row=1, col=1)

## -------------- ##
## Pressure Field ##
## -------------- ##
dark_rdbu_r = [[0.0, "#4a86c5"], [0.5, "#1a1a1a"], [1.0, "#d9605f"]]
fig.add_contour(x=x, y=y, z=Cp_plot.T,
                contours=dict(start=-3, end=1, size=0.25, coloring="fill"),
                line_width=0, colorscale=dark_rdbu_r, zmin=-3, zmax=1, zmid=0,
                colorbar=dict(title="Cp", x=0.42, len=1, thickness=14), hoverinfo="skip", row=1, col=1)

## ----------------------- ##
## free-stream streamlines ##
## ----------------------- ##
for y0 in np.linspace(-0.95, 0.95, 20):
    if abs(y0) > 1e-3:                     # skip y = 0, it runs straight into the stagnation point
        draw_streamline(np.array([-0.99, y0]))

## --------------------------------------- ##
## cylinder surface (dividing streamline)  ##
## --------------------------------------- ##
t = np.linspace(0, 2*np.pi, 200)
fig.add_scatter(x=R*np.cos(t), y=R*np.sin(t), mode="lines", line=dict(color="white", width=3), hoverinfo="skip", row=1, col=1)
fig.add_scatter(x=[-1, -R], y=[0, 0], mode="lines", line=dict(color="white", width=3), hoverinfo="skip", row=1, col=1)
fig.add_scatter(x=[R, 1],  y=[0, 0], mode="lines", line=dict(color="white", width=3), hoverinfo="skip", row=1, col=1)

## ----------------- ##
## stagnation points ##
## ----------------- ##
fig.add_scatter(x=[-R, R], y=[0, 0], mode="markers",
                marker=dict(symbol="star", size=16, color="black",
                            line=dict(color="white", width=1.5)), row=1, col=1)

## ------------ ##
## theta marker ##
## ------------ ##
th0 = np.deg2rad(50)                                  # angle shown in the sketch
ref = dict(mode="lines", hoverinfo="skip", showlegend=False, row=1, col=1)

fig.add_scatter(x=[0, R], y=[0, 0], line=dict(color="white", width=1, dash="dot"), **ref)                  # θ = 0 reference
fig.add_scatter(x=[0, R*np.cos(th0)], y=[0, R*np.sin(th0)], line=dict(color="white", width=1.5), **ref)   # radius to the surface
arc = np.linspace(0, th0, 30)
fig.add_scatter(x=0.4*R*np.cos(arc), y=0.4*R*np.sin(arc), line=dict(color="white", width=1.5), **ref)     # the arc

fig.add_annotation(x=0.55*R*np.cos(th0/2), y=0.55*R*np.sin(th0/2), text="θ", showarrow=False,
                   font=dict(color="white", size=16), row=1, col=1)

## --------------------- ##
## Cp across the surface ##
## --------------------- ##
theta = np.linspace(0, 2*np.pi, 361)
Cp_at = RegularGridInterpolator((x, y), Cp)            # Cp is indexed [x, y]
Cp_num = Cp_at(np.column_stack([R*np.cos(theta), R*np.sin(theta)]))
Cp_exact = 1 - 4*np.sin(theta)**2

fig.add_scatter(x=np.rad2deg(theta), y=Cp_exact, mode="lines", line=dict(color="white", width=2),
                name="exact", row=1, col=2)
fig.add_scatter(x=np.rad2deg(theta[::15]), y=Cp_num[::15], mode="markers",
                marker=dict(color="#d9605f", size=7), name="from field", row=1, col=2)

# sized for the page's column (about 730 px wide): the square flow field, the
# colour bar and the Cp graph then all have the same height
fig.update_layout(title="Non-lifting cylinder: uniform flow + doublet", width=730, height=400,
                  margin=dict(l=40, r=10, t=80, b=50),
                  showlegend=False,
                  xaxis=dict(range=[-1, 1], constrain="domain"),
                  yaxis=dict(range=[-1, 1], scaleanchor="x"))
fig.update_layout(template="plotly_dark",
                  paper_bgcolor="rgba(0,0,0,0)", plot_bgcolor="rgba(0,0,0,0)",
                  font_color="#e0e0e0")
fig.update_xaxes(fixedrange=True)          # no zooming or panning on any axis
fig.update_yaxes(fixedrange=True)
fig.update_layout(dragmode=False)          # dragging doesn't box-select either
fig.update_yaxes(range=[-1, 1], scaleanchor="x", constrain="domain", row=1, col=1)
fig.update_xaxes(title="θ [deg]", tickvals=[0, 90, 180, 270, 360], row=1, col=2)
fig.update_yaxes(title="Cp", row=1, col=2)
fig.show(config={"displayModeBar": False, "scrollZoom": False})
# page:hide-to

### Lifting flow over a cylinder
<!-- page:closed -->

_To be written._

In [27]:
# page:hide
# page:rerun=1

# page:slider (0.5, 5.0, 0.1)
Vinf = 1

# page:slider (0.1, 0.6, 0.05)
R = 0.6

# page:slider (0.5, 2, 0.1)
Gamma = 1

In [28]:
kappa = 2*np.pi*Vinf*R**2                            # doublet strength that gives a cylinder of radius R
const = -Gamma/(2*np.pi)*np.log(R)

phi_1, psi_1 = uniform(Vinf, 0)
phi_2, psi_2 = doublet(kappa)
phi_3, psi_3 = vortex(Gamma)

phi_t = phi_1 + phi_2 +  phi_3
psi_t = psi_1 + psi_2 + (psi_3 + const)

u =  ddy(psi_t)
v = -ddx(psi_t)

V  = np.hypot(u, v)
Cp = 1 - (V/Vinf)**2

# page:hide-from
inside  = np.hypot(X, Y) < R
Cp_plot = np.where(inside, np.nan, Cp)             # blank out the inside of the cylinder
psi_plot = np.where(inside, np.nan, psi_t)

fig = make_subplots(rows=1, cols=2, column_widths=[0.5, 0.5], horizontal_spacing=0.14,
                    subplot_titles=("Flow field", "Surface pressure"))

def draw_streamline(x0, width=1, **opts):
    xs, ys = streamline(x0, (u, v), (x, y), **opts)
    fig.add_scatter(x=xs, y=ys, mode="lines", line=dict(color="white", width=width),
                    hoverinfo="skip", showlegend=False, row=1, col=1)

## -------------- ##
## Pressure Field ##
## -------------- ##
dark_rdbu_r = [[0.0, "#4a86c5"], [0.5, "#1a1a1a"], [1.0, "#d9605f"]]
fig.add_contour(x=x, y=y, z=Cp_plot.T,
                contours=dict(start=-3, end=1, size=0.25, coloring="fill"),
                line_width=0, colorscale=dark_rdbu_r, zmin=-3, zmax=1, zmid=0,
                colorbar=dict(title="Cp", x=0.47, len=1.0, thickness=15),
                hoverinfo="skip", row=1, col=1)

## ----------------------- ##
## free-stream streamlines ##
## ----------------------- ##
for y0 in np.linspace(-0.95, 0.95, 20):
    draw_streamline(np.array([-0.99, y0]))

## ------------------------------------- ##
## dividing streamline (psi = 0) + body  ##
## ------------------------------------- ##
fig.add_contour(x=x, y=y, z=psi_plot.T, contours=dict(start=0, end=0, size=1, coloring="lines"),
                line=dict(color="white", width=3), colorscale=[[0, "white"], [1, "white"]],
                showscale=False, hoverinfo="skip", row=1, col=1)
t = np.linspace(0, 2*np.pi, 200)
fig.add_scatter(x=R*np.cos(t), y=R*np.sin(t), mode="lines", line=dict(color="white", width=3),
                hoverinfo="skip", showlegend=False, row=1, col=1)

## ----------------- ##
## stagnation points ##
## ----------------- ##
s = -Gamma/(4*np.pi*Vinf*R)                        # sin(theta_stag), Anderson: clockwise Gamma > 0
if abs(s) <= 1:
    th_s = np.arcsin(s)
    xs_stag = [R*np.cos(th_s), -R*np.cos(th_s)]
    ys_stag = [R*np.sin(th_s),  R*np.sin(th_s)]
    fig.add_scatter(x=xs_stag, y=ys_stag, mode="markers",
                    marker=dict(symbol="star", size=16, color="black",
                                line=dict(color="white", width=1.5)), row=1, col=1)
## ------------ ##
## theta marker ##
## ------------ ##
th0 = np.deg2rad(50)
ref = dict(mode="lines", hoverinfo="skip", showlegend=False, row=1, col=1)
fig.add_scatter(x=[0, R], y=[0, 0], line=dict(color="white", width=1, dash="dot"), **ref)
fig.add_scatter(x=[0, R*np.cos(th0)], y=[0, R*np.sin(th0)], line=dict(color="white", width=1.5), **ref)
arc = np.linspace(0, th0, 30)
fig.add_scatter(x=0.4*R*np.cos(arc), y=0.4*R*np.sin(arc), line=dict(color="white", width=1.5), **ref)
fig.add_annotation(x=0.55*R*np.cos(th0/2), y=0.55*R*np.sin(th0/2), text="θ", showarrow=False,
                   font=dict(color="white", size=16), row=1, col=1)

## ---------------- ##
## surface pressure ##
## ---------------- ##
theta = np.linspace(0, 2*np.pi, 361)
Cp_at  = RegularGridInterpolator((x, y), Cp)       # Cp is indexed [x, y]
Cp_num = Cp_at(np.column_stack([R*np.cos(theta), R*np.sin(theta)]))
Cp_exact = 1 - (2*np.sin(theta) + Gamma/(2*np.pi*R*Vinf))**2

fig.add_scatter(x=np.rad2deg(theta), y=Cp_exact, mode="lines", line=dict(color="white", width=2),
                name="exact", row=1, col=2)
fig.add_scatter(x=np.rad2deg(theta[::15]), y=Cp_num[::15], mode="markers",
                marker=dict(color="#d9605f", size=7), name="from field", row=1, col=2)

## ------ ##
## layout ##
## ------ ##
fig.update_layout(title="Lifting cylinder: uniform flow + doublet + vortex",
                  width=840, height=420, margin=dict(l=50, r=20, t=80, b=50),
                  legend=dict(x=0.99, y=0.02, xanchor="right"), dragmode=False)
fig.update_layout(template="plotly_dark",
                  paper_bgcolor="rgba(0,0,0,0)", plot_bgcolor="rgba(0,0,0,0)",
                  font_color="#e0e0e0")
fig.update_xaxes(fixedrange=True)          # no zooming or panning on any axis
fig.update_yaxes(fixedrange=True)
fig.update_layout(dragmode=False)          # dragging doesn't box-select either
fig.update_xaxes(range=[-1, 1], constrain="domain", fixedrange=True, row=1, col=1)
fig.update_yaxes(range=[-1, 1], scaleanchor="x", constrain="domain", fixedrange=True, row=1, col=1)
fig.update_xaxes(title="θ [deg]", range=[0, 360], tickvals=[0, 90, 180, 270, 360], fixedrange=True, row=1, col=2)
fig.update_yaxes(title="Cp", fixedrange=True, row=1, col=2)
fig.show(config={"displayModeBar": False, "scrollZoom": False})
# page:hide-to

## Panel Methods

<!-- page:closed -->
### Source Panel Methods

_To be written._

In [29]:
# import matplotlib.pyplot as plt
# import numpy as np

def cylinder(r, theta):
    return np.asarray([r*np.cos(theta),r*np.sin(theta)])

# Input Parameters
n_panels = 3
r        = 1
V_inf    = 1.0

# Cylinder
theta    = np.linspace(0,2*np.pi,n_panels+1)
P        = cylinder(r,theta)                      # panel vertices   shape: [{x,y}, n_panels+1]

# Panel Parameters
M  = (P[:,1:]+P[:,:-1])/2                         # panel midpoints  shape: [{x,y}, n_panels  ]
L  = np.linalg.norm(P[:,1:] - P[:,:-1], axis=0)   # panel lengths    shape: [       n_panels  ]
T  =  P[:,1:]-P[:,:-1]
T /= np.linalg.norm(T, axis=0)
Q  = np.asmatrix([[0,1],[-1,0]])
N  = np.asarray(Q@T)

# Parametric variables
s = np.linspace(0, 1, 100)                        # parameter along a panel

# Populate LHS matrix
A = np.zeros((n_panels, n_panels))
for i in range(n_panels):

    xi, yi = M[:,i]
    ni     = N[:,i]

    for j in range(n_panels):

        if i == j:
            A[i,j] = 0.5
            continue

        xj  = P[0,j] + s*(P[0,j+1] - P[0,j])
        yj  = P[1,j] + s*(P[1,j+1] - P[1,j])

        Dx  = xi - xj
        Dy  = yi - yj
        rij = np.hypot(Dx,Dy)

        ddx_lnrij = Dx/(rij*rij)
        ddy_lnrij = Dy/(rij*rij)
        ddn_lnrij = ni[0]*ddx_lnrij + ni[1]*ddy_lnrij

        A[i,j] = 1/(2*np.pi) * np.trapezoid(ddn_lnrij, s*L[j]) # you need to integrate from the start of the panel to the end of the panel!

# Generate RHS vector
b = -V_inf*N[0] # -V∞ · n

gamma = np.linalg.solve(A, b)

# plt.plot(P[0],P[1])
# plt.plot(M[0],M[1], linestyle='none', marker='s')
# plt.quiver(M[0], M[1], N[0], N[1], angles='xy', scale_units='xy', scale=3, color='tab:red', label='normal')
# plt.quiver(M[0], M[1], T[0], T[1], angles='xy', scale_units='xy', scale=3, color='tab:blue', label='tangent')
# plt.gca().set_aspect('equal')
# plt.show()


<!-- page:closed -->
### Vortex Panel Methods

_To be written._
